In [1]:
# Tự nạp lại raw_feature.py mỗi khi sửa, không cần restart kernel
%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from feature_edge import build, load_data, FEAT_COLS

In [2]:
# load_data() đã sort theo Timestamp + dựng src/dest; build() tính đặc trưng cửa sổ trượt 6h
# theo TỪNG GIAO DỊCH (nhân quả, chỉ nhìn về quá khứ) -> lấy phần split=="train" để profile skew
df = load_data()
train = df[df["split"] == "train"]

feat_all = build(df)
feat = feat_all.loc[feat_all["split"] == "train", FEAT_COLS]
print("feat shape:", feat.shape)

  5,078,345 giao dịch | 515,088 node | cửa sổ 6h
      500,000/5,078,345 dòng | cửa sổ đang giữ  500,000 dòng |  361,541 cặp
    1,000,000/5,078,345 dòng | cửa sổ đang giữ  201,605 dòng |  131,664 cặp
    1,500,000/5,078,345 dòng | cửa sổ đang giữ  184,959 dòng |   96,799 cặp
    2,000,000/5,078,345 dòng | cửa sổ đang giữ   48,290 dòng |   24,439 cặp
    2,500,000/5,078,345 dòng | cửa sổ đang giữ  117,946 dòng |   58,968 cặp
    3,000,000/5,078,345 dòng | cửa sổ đang giữ  116,537 dòng |   58,281 cặp
    3,500,000/5,078,345 dòng | cửa sổ đang giữ  117,527 dòng |   58,352 cặp
    4,000,000/5,078,345 dòng | cửa sổ đang giữ  117,593 dòng |   57,108 cặp
    4,500,000/5,078,345 dòng | cửa sổ đang giữ  161,272 dòng |   88,438 cặp
    5,000,000/5,078,345 dòng | cửa sổ đang giữ   48,020 dòng |   24,973 cặp
feat shape: (3046861, 18)


In [ ]:
fx_gap    = np.log1p(df["Amount Received"]) - np.log1p(df["Amount Paid"])
cross_ccy = df["Receiving Currency"] != df["Payment Currency"]
y         = df["Is Laundering"]
max_abs_gap_same = np.abs(fx_gap[~cross_ccy]).max()
print(f"[1] Dòng cùng loại tiền : {(~cross_ccy).sum():>9,} ({(~cross_ccy).mean()*100:.2f}%)")
print(f"    max|fx_gap|         : {max_abs_gap_same:.10f}")
assert max_abs_gap_same == 0.0, "Có dòng cùng loại tiền mà fx_gap != 0"
nz = fx_gap != 0
print(f"\n[2] Dòng có fx_gap != 0  : {nz.sum():>9,} ({nz.mean()*100:.2f}%)")
print(f"    Dòng cross-currency  : {cross_ccy.sum():>9,} ({cross_ccy.mean()*100:.2f}%)")
assert (nz & ~cross_ccy).sum() == 0, "fx_gap != 0 nhưng không phải cross-currency"
n_pos_nz = int(y[nz].sum())
print(f"\n[3] Nhãn dương trong tập fx_gap != 0 : {n_pos_nz}")
print(f"    Nhãn dương toàn bộ dataset       : {int(y.sum()):,}")
assert n_pos_nz == 0, "Tập fx_gap != 0 có chứa nhãn dương"
print("\n[4] Phân rã theo split:")
rep = (pd.DataFrame({"split": df["split"], "gap_nonzero": nz.astype(int), "y": y})
         .groupby(["split", "gap_nonzero"])["y"].agg(n="size", n_pos="sum"))
rep["pos_%"] = (rep["n_pos"] / rep["n"] * 100).round(4)
print(rep.to_string())

print(f"\n=> fx_gap chỉ mang thông tin trên {nz.mean()*100:.2f}% số dòng,")
print(f"   và toàn bộ {nz.sum():,} dòng đó đều có nhãn ÂM.")
print("   Không tách được positive/negative ở bất kỳ đâu -> LOẠI (cùng với recv_paid_log).")

In [ ]:
# ── Thực nghiệm: amt_ratio có phụ thuộc hoàn toàn vào cặp tiền không? ──
# Chạy trên giao dịch thô (train), vì amt_ratio là đại lượng của TỪNG giao dịch.
t = train[train["Amount Received"] > 0].copy()
t["amt_ratio"] = t["Amount Paid"] / t["Amount Received"]
t["cur_pair"]  = t["Payment Currency"].astype(str) + "->" + t["Receiving Currency"].astype(str)

g = t.groupby("cur_pair")["amt_ratio"]
stats = pd.DataFrame({"count": g.count(), "mean": g.mean(), "std": g.std()})
stats["cv"] = stats["std"] / stats["mean"]          # hệ số biến thiên: bỏ ảnh hưởng độ lớn tỷ giá
stats = stats[stats["count"] >= 30].sort_values("cv", ascending=False)

print(stats.head(15).round(4))
med_cv = stats["cv"].median()
print(f"\nCV trung vị = {med_cv:.4f}")
print("=> amt_ratio bị cặp tiền quyết định hoàn toàn → TRÙNG currency proportion, nên BỎ."
      if med_cv < 0.01 else
      "=> amt_ratio còn dao động trong cùng cặp tiền → có thông tin mới, nên GIỮ (nhớ log1p).")

                              count        mean         std      cv
cur_pair                                                           
US Dollar->Bitcoin              833  11901.7459   1863.0107  0.1565
Yuan->Bitcoin                    80  74599.3660  10083.1622  0.1352
Canadian Dollar->Swiss Franc     36      1.4505      0.0955  0.0658
Euro->Bitcoin                   142  10144.1952    550.0176  0.0542
Ruble->US Dollar                 88     77.4282      3.7126  0.0479
Australian Dollar->US Dollar     79      1.4202      0.0661  0.0465
Brazil Real->US Dollar           44      5.6774      0.2040  0.0359
UK Pound->Bitcoin                59   9323.6592    290.8838  0.0312
Yen->US Dollar                  581    105.1980      3.0305  0.0288
Yuan->Brazil Real                31      1.1801      0.0334  0.0283
Mexican Peso->US Dollar         162     21.1828      0.5414  0.0256
Canadian Dollar->US Dollar      318      1.3175      0.0257  0.0195
Swiss Franc->Euro                44      1.0689 

In [ ]:
# Profile mọi cột để TỰ quyết định nhóm (thay cho việc hardcode ratio_cols/special)
profile = pd.DataFrame({
    "min":     feat.min(),
    "max":     feat.max(),
    "pct_neg": (feat < 0).mean() * 100,        # % giá trị âm -> >0 nghĩa là không dùng được log1p
    "in_0_1":  (feat.min() >= 0) & (feat.max() <= 1),  # True = ratio nằm gọn trong [0,1]
    "skew":    feat.skew(),
}).round(3)

print("== Cột nằm trong [0,1] (ứng viên ratio_cols) ==")
print(profile.index[profile["in_0_1"]].tolist())
print("\n== Cột có giá trị âm (ứng viên special) ==")
print(profile.index[profile["pct_neg"] > 0].tolist())
print("\n== Bảng đầy đủ ==")
profile.sort_values("skew", ascending=False)

== Cột nằm trong [0,1] (ứng viên ratio_cols) ==
['cross_ccy_ratio', 'round_ratio', 'is_cross_bank', 'is_self_loop', 'is_edge_mule']

== Cột có giá trị âm (ứng viên special) ==
[]

== Bảng đầy đủ ==


,min,max,pct_neg,in_0_1,skew
std_paid,0.0,3.293114e+11,0.0,False,496.384
max_paid,0.0,6.260355e+11,0.0,False,467.428
total_paid,0.0,1.013024e+12,0.0,False,430.206
mean_paid,0.0,3.376745e+11,0.0,False,415.677
min_paid,0.0,9.876640e+10,0.0,False,253.144
round_ratio,0.0,1.000000e+00,0.0,True,195.978
is_edge_mule,0.0,1.000000e+00,0.0,True,20.520
cross_ccy_ratio,0.0,1.000000e+00,0.0,True,11.542
num_tx,1.0,4.900000e+01,0.0,False,2.373
tx_per_day,0.4,9.800000e+00,0.0,False,1.877


In [ ]:
# Bảng skew: thô vs sau log1p
ratio_cols=[]
for i in profile.index:
    if profile.loc[i,"in_0_1"]==True:
        ratio_cols.append(i)
cand   = [c for c in feat.columns if c not in ratio_cols]              #cand là cột chứa cột cần xử lý
nonneg = [c for c in cand if feat[c].min() >= 0]                                 # log1p chỉ hợp lệ khi >= 0
report = pd.DataFrame({
    "min":        feat[cand].min(),
    "skew_raw":   feat[cand].skew(),
    "skew_log1p": np.log1p(feat[nonneg]).skew(),
}).sort_values("skew_raw", ascending=False)

report.round(2)

,min,skew_raw,skew_log1p
std_paid,0.0,496.38,0.90
max_paid,0.0,467.43,0.17
total_paid,0.0,430.21,0.05
mean_paid,0.0,415.68,0.20
min_paid,0.0,253.14,0.41
num_tx,1.0,2.37,1.25
tx_per_day,0.4,1.88,1.01
active_day,1.0,1.42,1.32


In [ ]:
# Đề xuất cột log1p (lệch phải mạnh + không âm). Vẫn nên nhìn skew_log1p để loại cột bị kéo quá đà.
log1p_cols = report.index[(report["skew_raw"] > 1) & (report["min"] >= 0)].tolist()
print("Đề xuất log1p_cols:")
for c in log1p_cols:
    print(f"  {c:18s} skew_raw={report.loc[c,'skew_raw']:.2f} -> skew_log1p={report.loc[c,'skew_log1p']:.2f}")

Đề xuất log1p_cols:
  std_paid           skew_raw=496.38 -> skew_log1p=0.90
  max_paid           skew_raw=467.43 -> skew_log1p=0.17
  total_paid         skew_raw=430.21 -> skew_log1p=0.05
  mean_paid          skew_raw=415.68 -> skew_log1p=0.20
  min_paid           skew_raw=253.14 -> skew_log1p=0.41
  num_tx             skew_raw=2.37 -> skew_log1p=1.25
  tx_per_day         skew_raw=1.88 -> skew_log1p=1.01
  active_day         skew_raw=1.42 -> skew_log1p=1.32


In [ ]:
col = "pair_cnt"  # tương đương num_tx cũ: số giao dịch của cặp (src,dest) TRONG cửa sổ 6h
plt.boxplot([feat[col], np.log1p(feat[col])], labels=["raw", "log1p"])
plt.title(f"{col} boxplot"); plt.show()